# CPEC detection: RetinaNet training v2
Same model as the original notebook, with these changes:
1. **Augmentation that actually works for RetinaNet.** Flips and 90° rotations move the box targets with the image, and per-channel colour jitter simulates stain variation. The original generator only applied contrast, and per-tile normalization cancelled most of it.
2. **Optional new anchors.** Re-encodes the pickled targets for more anchor scales and ratios.
3. **Learning-rate schedule.** Reduces the learning rate when validation loss plateaus.
4. **Checkpoint chosen by detection F1**, not by eye.

Run `CPEC detection evaluation.ipynb` on the old model first so you have a baseline to compare against.

In [ ]:
import pickle, glob
import numpy as np, matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import Input
from jarvis.train.box import BoundingBox
import biondi
import cpec_tools as ct

In [ ]:
training_dataset_dir = ''
validation_dataset_dir = ''
with open(training_dataset_dir, 'rb') as f: x = pickle.load(f)
with open(validation_dataset_dir, 'rb') as f: v = pickle.load(f)

bb_orig = BoundingBox(image_shape=(256, 256), classes=1, c=[3], anchor_shapes=[16],
                      anchor_scales=[1], anchor_ratios=[1], iou_upper=0.5, iou_lower=0.2)

## Anchors
Leave `CHANGE_ANCHORS = False` to keep the original anchors. Set it to `True` only if the box-size report in the evaluation notebook shows many cells that 16 px square anchors can't match.

Change one thing at a time (augmentation first, then anchors) so you can tell what helped.

In [ ]:
CHANGE_ANCHORS = False

if CHANGE_ANCHORS:
    scales, ratios = [0.8, 1, 1.26, 1.6], [1]      # keep ratios=[1] to allow 90-degree rotation augmentation
    bb = BoundingBox(image_shape=(256, 256), classes=1, c=[3], anchor_shapes=[16],
                     anchor_scales=scales, anchor_ratios=ratios, iou_upper=0.5, iou_lower=0.3)
    x = ct.reencode_targets(x, bb_orig, bb)
    v = ct.reencode_targets(v, bb_orig, bb)
    A = len(scales) * len(ratios)
else:
    bb, A = bb_orig, 1

## Augmentation
`calibrate_geometry` flips real targets and checks that the decoded boxes land in the mirrored positions. Any transform that isn't exact is switched off automatically.

In [ ]:
layout, ok_ops = ct.calibrate_geometry(bb, x)
if CHANGE_ANCHORS and any(r != 1 for r in ratios):
    ok_ops = [o for o in ok_ops if o != 'transpose']   # transposing would swap tall/wide anchors
print('augmentations in use:', ok_ops)

xgen = ct.AugmentedRetinaNetGenerator(data=x, batch_size=16, layout=layout, geo_ops=ok_ops, colour=True)
vgen = biondi.dataset.TrainingGenerator(data=v, batch_size=16, retinanet=True, validation=True)

In [ ]:
# sanity check: augmented tile with its decoded target boxes
b = xgen[0]
boxes = ct.ground_truth_boxes(bb, {k: val for k, val in b.items() if k != 'dat'})
ct.show_tiles({'dat': b['dat']}, boxes, boxes, range(4))

## Model

In [ ]:
inputs = bb.get_inputs({'dat': Input(name='dat', shape=(None, 256, 256, 3)),
                        'cls-c3': Input(name='cls-c3_y', shape=(None, 32, 32, A)),
                        'reg-c3': Input(name='reg-c3_y', shape=(None, 32, 32, 4 * A))}, Input)
rn_model = biondi.models.retinanet_resnet50_3d(inputs, K=1, A=A, feature_maps=('c3'), lr=2e-4, filter_ratio=1)

## Train

In [ ]:
chckpnt_dir = ''
callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=chckpnt_dir + 'model_epoch-{epoch:02d}_val_loss-{val_loss:.4f}'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
]
rn_hist = rn_model.fit(x=xgen, epochs=60, validation_data=vgen, workers=16, max_queue_size=100,
                       verbose=1, callbacks=callbacks)
plt.plot(rn_hist.history['loss'], label='train'); plt.plot(rn_hist.history['val_loss'], label='val')
plt.legend(); plt.show()

## Pick the checkpoint by validation F1

In [ ]:
results = ct.evaluate_checkpoints(sorted(glob.glob(chckpnt_dir + 'model_epoch*')), bb, v)
best = results.loc[results.f1.idxmax()]
print(best)

Then run `CPEC detection evaluation.ipynb` on `best.checkpoint` with the **test** set and compare against the baseline. Use `best.score_thr` as the detection threshold for whole-slide inference.